# MevzuatTool — BGE-M3 Embedding (Colab)

**Amaç:** korpus.jsonl'deki maddeleri BGE-M3 ile embed et → dense + sparse vektörleri kaydet → PC'ye indir.

**Mimari:** Ağır iş (embedding) burada (Colab T4 GPU). Hafif iş (Qdrant + arama) PC'de.

**ÖNCE:** Runtime menüsü → Change runtime type → **T4 GPU** seç. (Ücretsiz.)

> Bu bir **kullanım** (inference), fine-tune DEĞİL → kapsam içi (retrieval). Model olduğu gibi kullanılıyor.

## 1. GPU kontrolü
T4 GPU seçili mi doğrula. `Tesla T4` görmelisin (16 GB VRAM).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. FlagEmbedding kur
BGE-M3'ü çalıştıran kütüphane. Colab'da torch zaten kurulu (GPU'lu) → sadece FlagEmbedding gerek.
(~1-2 dk sürer.)

In [ ]:
!pip install -q FlagEmbedding

## 3. Google Drive'ı bağla
korpus.jsonl'i Drive'a yükledikten sonra buradan okuruz.

**ÖNCE:** korpus.jsonl'i Google Drive'a yükle (örn. `MyDrive/mevzuat/korpus.jsonl`).
Çalıştırınca izin isteyecek — onayla.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Drive'daki korpus yolu — KENDİ yoluna göre düzelt:
KORPUS_YOLU = '/content/drive/MyDrive/mevzuat/korpus.jsonl'

import os
print('Dosya var mı:', os.path.exists(KORPUS_YOLU))
if os.path.exists(KORPUS_YOLU):
    print('Boyut:', round(os.path.getsize(KORPUS_YOLU)/1e6, 1), 'MB')

## 4. Korpusu yükle

**TEST için:** `LIMIT = 500` (ilk 500 madde). Çalıştığını görünce `LIMIT = None` yapıp tam korpusu (31.416) embed et.

Her madde için embed edilecek metin: **`passage: {text}`**.
- `passage:` öneki BGE-M3 için değil ama e5-tarzı retrieval'da yaygın; BGE-M3'te opsiyonel.
- Biz **ham text** kullanacağız (BGE-M3 önek istemez). Not: madde başlığı text'in içinde zaten var.

In [ ]:
import json

LIMIT = 500   # TEST: 500 | TAM: None

kayitlar = []
with open(KORPUS_YOLU, encoding='utf-8') as f:
    for i, line in enumerate(f):
        if LIMIT and i >= LIMIT:
            break
        r = json.loads(line)
        kayitlar.append(r)

print(f'{len(kayitlar)} madde yüklendi')

# Embed edilecek metinler (ham text) + id'ler + payload (metadata)
ids = [r['id'] for r in kayitlar]
metinler = [r['text'] for r in kayitlar]
payloadlar = [
    {
        'kanun_no': r['metadata']['kanun_no'],
        'kanun_ad': r['metadata']['kanun_ad'],
        'madde_no': r['metadata']['madde_no'],
        'yurutluk': r['metadata']['yurutluk'],
    }
    for r in kayitlar
]
print('Örnek metin:', metinler[0][:100])

## 5. BGE-M3 modelini yükle
İlk çağrıda HuggingFace'ten ~2.5 GB iner (Colab'ın hızlı bağlantısıyla ~1 dk).

`use_fp16=True` → yarı-hassasiyet (float16), GPU'da hız + bellek tasarrufu (T4 için ideal).

In [ ]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
print('Model yüklendi.')

## 6. Embed et — dense + sparse

BGE-M3'ün **multi-functionality**si: tek çağrıda hem dense hem sparse üretir.
- `return_dense=True` → 1024 boyutlu anlam vektörü
- `return_sparse=True` → kelime-ağırlık sözlüğü (BM25 benzeri, hybrid için)
- `return_colbert_vecs=False` → ColBERT'i kullanmıyoruz (MVP'de gerek yok)

`batch_size` T4'te 12 rahat. (Senin 4GB laptop'ta bu mümkün olmazdı — Colab'ın avantajı.)

In [ ]:
import time
t0 = time.time()

cikti = model.encode(
    metinler,
    batch_size=12,
    max_length=8192,          # BGE-M3'ün tam bağlamı — uzun madde kesilmez
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=False,
)

print(f'Embed süresi: {time.time()-t0:.1f} sn ({len(metinler)} madde)')
print('Dense şekli:', cikti['dense_vecs'].shape)   # (N, 1024)
print('Sparse örnek (ilk maddenin kelime ağırlıkları):')
print(dict(list(cikti['lexical_weights'][0].items())[:5]))

## 7. Vektörleri kaydet (PC'ye indirmek için)

- **dense** → `.npy` (numpy dizisi, verimli)
- **sparse** → JSON (kelime→ağırlık sözlükleri)
- **ids + payload** → JSON (Qdrant'a yüklerken lazım)

Sparse ağırlıkların anahtarları token-id (sayı) → JSON için string'e çeviriyoruz.

In [ ]:
import numpy as np
import json

CIKTI_DIR = '/content/drive/MyDrive/mevzuat/'

# 1) dense
np.save(CIKTI_DIR + 'dense.npy', cikti['dense_vecs'])

# 2) sparse (token-id -> agirlik); anahtarlari str yap (JSON geregi)
sparse_json = [
    {str(k): float(v) for k, v in w.items()}
    for w in cikti['lexical_weights']
]
with open(CIKTI_DIR + 'sparse.json', 'w', encoding='utf-8') as f:
    json.dump(sparse_json, f)

# 3) ids + payload
with open(CIKTI_DIR + 'meta.json', 'w', encoding='utf-8') as f:
    json.dump({'ids': ids, 'payloadlar': payloadlar}, f, ensure_ascii=False)

print('Kaydedildi ->', CIKTI_DIR)
print('  dense.npy   :', cikti['dense_vecs'].shape)
print('  sparse.json :', len(sparse_json), 'madde')
print('  meta.json   :', len(ids), 'id + payload')
print('Bu 3 dosyayi Drive-den PC-ye indir, Qdrant-a yukleyecegiz.')

## 8. (Opsiyonel) Hızlı doğrulama — dense benzerlik testi

Embedding çalışıyor mu? Bir sorguyu embed edip ilk maddelere cosine benzerliğine bak.
(Asıl arama PC'de Qdrant ile olacak; bu sadece "embedding mantıklı mı" kontrolü.)

In [ ]:
sorgu = 'bir memur hangi durumlarda görevinden çıkarılır'
sq = model.encode([sorgu], return_dense=True, return_sparse=False)['dense_vecs'][0]

# cosine (dense_vecs normalize geldiği için nokta çarpım = cosine)
skorlar = cikti['dense_vecs'] @ sq
top = np.argsort(-skorlar)[:5]

print(f'Sorgu: {sorgu}\n')
for i in top:
    print(f"  {skorlar[i]:.3f} | {ids[i]} | {metinler[i][:70]}")